# Sentinel-1 L0 → SLC Batch Order and S3 Upload

Submits a [Batch Order](https://documentation.dataspace.copernicus.eu/APIs/On-Demand%20Production%20API.html)
for multiple Sentinel-1 L0 scenes via the CDSE On-Demand Production API, then downloads, unzips,
and uploads each SLC product to S3. See
[`sentinel1_l0_to_slc_ondemand.ipynb`](./sentinel1_l0_to_slc_ondemand.ipynb) for the single-scene
walkthrough, and [`search_ew_l0_scenes.ipynb`](./search_ew_l0_scenes.ipynb) to build
`L0_PRODUCTS` from an AOI.

**Steps:** authenticate → check L0 availability (LTA) → submit or resume batch order → poll until
complete → download/unzip/upload. Designed to be re-run top to bottom — it won't resubmit a
batch order that already exists.

## Configuration

Set the L0 product list, batch name, and S3 destination. Credentials come from
`CDSE_USERNAME`/`CDSE_PASSWORD` env vars (or enter directly).

In [ ]:
import os
import json
from datetime import date
from dotenv import load_dotenv

try:
    # load credentials from root .env file
    load_dotenv("../../.env")
except:
    print("Could not find .env file with credentials.")

# --- INPUT -------------------------------------------------------------------
# List of L0 products to process together as a single Batch Order.
# See search_ew_l0_scenes.ipynb to build this list from an AOI, e.g. via:
# L0_PRODUCTS = json.load(open("EW_L0_scene_list.txt"))
L0_PRODUCTS = [line.strip() for line in open("EW_L0_scene_list.txt") if line.strip()]
# L0_PRODUCTS = [
#     "S1A_EW_RAW__0SSH_20220131T034822_20220131T034930_041699_04F623_CCA2.SAFE",
# ]

# A short label for the batch order (visible in the ODP portal)
BATCH_ORDER_NAME = f"slc_batch_{date.today().strftime("%Y%m%d")}_{len(L0_PRODUCTS)}_scenes"
# BATCH_ORDER_NAME = "slc_batch_20260812_8_scenes"

# Where to save downloaded/unzipped files locally before upload
OUTPUT_DIR = "."

# Number of scenes to download/unzip/upload concurrently in section 6. Reduce this if you
# hit rate limits or connection errors against CDSE object storage or S3.
MAX_WORKERS = 4

# S3 destination for the unzipped SLC products produced by the batch
S3_BUCKET = "deant-data-public-dev"
S3_PROJECT_FOLDER = "s1_ew_slc"
S3_SCENE_UPLOAD_FOLDER = f"{S3_PROJECT_FOLDER}/data"
S3_SCENE_TRACKING_UPLOAD_FOLDER = f"{S3_PROJECT_FOLDER}/tracking"

# Credentials — prefer env vars so they are not committed
CDSE_USERNAME = os.environ.get("CDSE_LOGIN", "")  # or set directly: "your@email.com"
CDSE_PASSWORD = os.environ.get("CDSE_PASSWORD", "")  # or set directly: "yourpassword"
# -----------------------------------------------------------------------------

WORKFLOW_NAME = "Sentinel-1-L0-EW_SLC__1S"
BASE_URL = "https://odp.dataspace.copernicus.eu/odata/v1"
CATALOGUE_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1"
ZIPPER_URL = "https://zipper.dataspace.copernicus.eu/odata/v1"
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

print(f"Scenes     : {len(L0_PRODUCTS)}")
print(f"Batch name : {BATCH_ORDER_NAME}")
print(f"Output dir : {os.path.abspath(OUTPUT_DIR)}")
print(f"S3 target  : s3://{S3_BUCKET}/{S3_SCENE_UPLOAD_FOLDER}")

## 1. Authenticate

In [ ]:
import json
import requests
import getpass
import time as _time

if not CDSE_USERNAME:
    CDSE_USERNAME = input("Copernicus username (email): ")
if not CDSE_PASSWORD:
    CDSE_PASSWORD = getpass.getpass("Copernicus password: ")


def get_token(username: str, password: str) -> str:
    resp = requests.post(
        TOKEN_URL,
        data={
            "client_id": "cdse-public",
            "username": username,
            "password": password,
            "grant_type": "password",
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    if not resp.ok:
        try:
            detail = resp.json()
        except Exception:
            detail = resp.text
        raise RuntimeError(
            f"Authentication failed ({resp.status_code}): {detail}\n\n"
            "Check that:\n"
            "  1. Your Copernicus Data Space account is verified at https://dataspace.copernicus.eu\n"
            "  2. You are using your registration email + password (not a Google/GitHub SSO login)\n"
            "  3. There are no leading/trailing spaces in your credentials"
        )
    data = resp.json()
    return data["access_token"], data.get("expires_in", 600)


TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
TOKEN_ACQUIRED_AT = _time.monotonic()


def auth_headers() -> dict:
    """Return headers with a fresh token, refreshing if within 60s of expiry."""
    global TOKEN, TOKEN_ACQUIRED_AT, TOKEN_EXPIRES_IN
    if _time.monotonic() - TOKEN_ACQUIRED_AT > (TOKEN_EXPIRES_IN - 60):
        print("Refreshing token...")
        TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
        TOKEN_ACQUIRED_AT = _time.monotonic()
    return {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


print(f"Authenticated successfully. Token valid for {TOKEN_EXPIRES_IN}s.")

## 2. Check L0 Input Product Availability (Online / LTA)

CDSE's order service is documented to pull offline inputs from LTA itself, so this is a
diagnostic step rather than a hard requirement. For each scene in `L0_PRODUCTS`: zero catalogue
results means it isn't in the archive at all (report to CDSE support); `Online: false` triggers a
restore attempt; `Online: true` needs nothing. Restores are triggered for all offline scenes up
front so the waits below run concurrently.

In [ ]:
from datetime import datetime


def get_product_by_name(name: str) -> dict:
    resp = requests.get(
        f"{CATALOGUE_URL}/Products",
        params={
            "$filter": f"Name eq '{name}'",
            "$select": "Id,Name,Online,ContentLength",
        },
        headers=auth_headers(),
        timeout=30,
    )
    resp.raise_for_status()
    products = resp.json().get("value", [])
    if not products:
        raise RuntimeError(
            f"Product '{name}' not found in the CDSE catalogue — it may have been purged "
            "from the archive entirely rather than simply being offline in the LTA."
        )
    return products[0]


def trigger_lta_retrieval(product_id: str) -> int:
    """GET the product's $value URL to trigger an LTA restore; connection closes right
    after headers arrive, before the (potentially multi-GB) body streams."""
    with requests.get(
        f"{ZIPPER_URL}/Products({product_id})/$value",
        headers=auth_headers(),
        stream=True,
        timeout=60,
    ) as r:
        return r.status_code


def wait_for_online(
    product_id: str, poll_interval_seconds: int = 300, timeout_hours: float = 24.0
) -> None:
    deadline = _time.monotonic() + timeout_hours * 3600
    while True:
        resp = requests.get(
            f"{CATALOGUE_URL}/Products({product_id})",
            params={"$select": "Online"},
            headers=auth_headers(),
            timeout=30,
        )
        resp.raise_for_status()
        online = resp.json()["Online"]
        print(f"[{datetime.utcnow().strftime('%H:%M:%S')} UTC]  Online: {online}")
        if online:
            return
        if _time.monotonic() > deadline:
            raise TimeoutError(
                f"Product {product_id} did not come online within {timeout_hours}h "
                "of triggering LTA retrieval."
            )
        _time.sleep(poll_interval_seconds)

In [ ]:
l0_products_info = {}
for product_name in L0_PRODUCTS:
    info = get_product_by_name(product_name)
    l0_products_info[product_name] = info
    print(f"{product_name}: Online={info['Online']}")
    if not info["Online"]:
        print(f"  Triggering LTA retrieval for {product_name}...")
        trigger_lta_retrieval(info["Id"])

# Wait for any offline products to come back online (retrieval was triggered for all of
# them above, so the waits below overlap rather than compounding).
for product_name, info in l0_products_info.items():
    if not info["Online"]:
        print(f"\nWaiting for {product_name} to come online...")
        wait_for_online(info["Id"])
        print(f"{product_name} is now online.")

print("\nAll scenes in L0_PRODUCTS are online and ready to order.")

## 3. Submit or Resume Batch Order

Looks up `BATCH_ORDER_NAME` first; reuses it if found, otherwise submits a new one. Safe to
re-run top to bottom without resubmitting.

In [ ]:
def get_batch_order_by_name(name: str) -> dict | None:
    """Look up a previously-submitted BatchOrder by Name; None if none exists."""
    resp = requests.get(
        f"{BASE_URL}/BatchOrder",
        params={"$filter": f"Name eq '{name}'"},
        headers=auth_headers(),
        timeout=30,
    )
    resp.raise_for_status()
    orders = resp.json().get("value", [])
    if not orders:
        return None
    orders.sort(key=lambda o: o.get("SubmissionDate", ""), reverse=True)
    return orders[0]


def submit_batch_order(name: str, l0_products: list[str]) -> dict:
    batch_payload = {
        "Name": name,
        "WorkflowName": WORKFLOW_NAME,
        "IdentifierList": l0_products,
        "WorkflowOptions": [],
        "Priority": 1,
    }
    resp = requests.post(
        f"{BASE_URL}/BatchOrder/OData.CSC.Order",
        headers=auth_headers(),
        json=batch_payload,
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json().get("value", resp.json())


batch_order = get_batch_order_by_name(BATCH_ORDER_NAME)
if batch_order is None:
    print(f"No existing batch order named '{BATCH_ORDER_NAME}' — submitting.")
    batch_order = submit_batch_order(BATCH_ORDER_NAME, L0_PRODUCTS)
else:
    print(f"Found existing batch order '{BATCH_ORDER_NAME}' — reusing.")

BATCH_ORDER_ID = batch_order["Id"]
print(f"  ID     : {BATCH_ORDER_ID}")
print(f"  Status : {batch_order.get('Status')}")
print(f"  Name   : {batch_order.get('Name')}")

## 4. Poll Batch Order Until Complete

Polls every `POLL_INTERVAL_SECONDS`. Section 5 still checks each item individually even if the
batch doesn't complete cleanly, since some scenes may have succeeded. Interrupt the kernel to
stop polling.

In [ ]:
import time

POLL_INTERVAL_SECONDS = 60
TERMINAL_STATUSES = {"completed", "failed", "cancelled"}


def get_batch_order_status(batch_order_id: str) -> dict:
    r = requests.get(
        f"{BASE_URL}/BatchOrder({batch_order_id})",
        headers=auth_headers(),
        timeout=30,
    )
    if not r.ok:
        print(f"  Warning: {r.status_code} — {r.text[:200]}")
        r.raise_for_status()
    data = r.json()
    return data.get("value", data)


while True:
    batch_info = get_batch_order_status(BATCH_ORDER_ID)
    batch_status = str(batch_info.get("Status", "unknown")).lower()
    print(f"[{datetime.utcnow().strftime('%H:%M:%S')} UTC]  Batch status: {batch_status}")

    if batch_status in TERMINAL_STATUSES:
        break

    time.sleep(POLL_INTERVAL_SECONDS)

if batch_status != "completed":
    print("\nFull batch order details:")
    print(json.dumps(batch_info, indent=2))
    print(
        "\nBatch did not complete cleanly — proceeding to section 5 anyway, since "
        "individual scenes may still have succeeded."
    )
else:
    print("\nBatch order completed.")

## 5. List Batch Ordered Items

In [ ]:
def list_batch_order_items(batch_order_id: str) -> list[dict]:
    resp = requests.get(
        f"{BASE_URL}/BatchOrder({batch_order_id})/Products",
        headers=auth_headers(),
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    items = data.get("value", data)
    if not isinstance(items, list):
        items = [items]
    return items


batch_items = list_batch_order_items(BATCH_ORDER_ID)
print(f"{len(batch_items)} item(s) in batch order {BATCH_ORDER_ID}:")
print(json.dumps(batch_items, indent=2))

## 6. Download, Unzip, and Upload Results to S3

Each processed item's output is a .zip file. Inside that .zip is the SLC .zip we're actually
interested in. The SLC name isn't readily available up front, so we peek into the outer .zip to
fetch it, then check whether that file has already been uploaded — this is what makes resuming
safe.

Items are processed concurrently across `MAX_WORKERS` threads (set in the Configuration cell
above). For each item the code will:
1. Peek into the .zip produced by the L0 → L1 processor to get the SLC name.
2. Check if the SLC has already been uploaded to S3.
3. If yes, move on to the next item. Otherwise, download and unzip the SLC, then upload it along
   with a tracking file to S3. The tracking file contains useful information about the SLC
   product — e.g. when it was processed and the name of the input L0 scene.

Requires AWS credentials in the environment — see
`sar_pipeline.utils.aws.check_aws_environment_credentials`.

In [ ]:
import zipfile
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import boto3
from botocore.exceptions import ClientError

from sar_pipeline.utils.aws import check_aws_environment_credentials

check_aws_environment_credentials(verbose=True)
s3_client = boto3.client("s3")

_print_lock = threading.Lock()


def log(message: str) -> None:
    """Thread-safe print so parallel workers don't interleave mid-line."""
    with _print_lock:
        print(message)


class _HTTPRangeFile:
    """Read+seek file-like object backed by HTTP Range requests, so zipfile can read just
    the central directory of a remote zip without downloading the body."""

    def __init__(self, url: str, size: int):
        self._url = url
        self._size = size
        self._pos = 0

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self._pos

    def seek(self, offset: int, whence: int = 0) -> int:
        if whence == 0:
            self._pos = offset
        elif whence == 1:
            self._pos += offset
        elif whence == 2:
            self._pos = self._size + offset
        else:
            raise ValueError(f"Unsupported whence: {whence}")
        return self._pos

    def read(self, size: int = -1) -> bytes:
        if size is None or size < 0:
            end = self._size - 1
        else:
            end = min(self._pos + size, self._size) - 1
        if self._pos > end:
            return b""
        resp = requests.get(self._url, headers={"Range": f"bytes={self._pos}-{end}"}, timeout=60)
        if resp.status_code not in (200, 206):
            resp.raise_for_status()
        data = resp.content
        self._pos += len(data)
        return data


def peek_remote_inner_zip_name(download_link: str, known_size: int | None = None) -> str:
    """Return the inner filename inside a remote order .zip via Range requests against its
    central directory -- no download, no unzip. Raises if ranges aren't supported or the
    archive doesn't hold exactly one member, so the caller can fall back to download+unzip."""
    probe = requests.get(download_link, headers={"Range": "bytes=0-0"}, timeout=30)
    if probe.status_code != 206:
        raise RuntimeError(
            f"Server did not honor a Range request (status {probe.status_code}); "
            "cannot peek without downloading."
        )
    content_range = probe.headers.get("Content-Range", "")
    try:
        total_size = int(content_range.rsplit("/", 1)[-1])
    except (ValueError, IndexError):
        if not known_size:
            raise RuntimeError(f"Could not parse total size from Content-Range: {content_range!r}")
        total_size = known_size

    remote_file = _HTTPRangeFile(download_link, total_size)
    with zipfile.ZipFile(remote_file) as zf:
        names = zf.namelist()
    inner_zips = [n for n in names if n.lower().endswith(".zip")]
    if len(inner_zips) != 1:
        raise RuntimeError(f"Expected exactly one inner .zip, found: {names}")
    return inner_zips[0]


def download_batch_item(download_link: str, local_path: str) -> None:
    """Download a batch item's product from its pre-signed Swift `DownloadLink`."""
    with requests.get(download_link, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)


def unzip_single_inner_zip(zip_path: str, extract_dir: str) -> str:
    """Extract the order's wrapper .zip and return the path to the inner product .zip."""
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        inner_zips = [n for n in names if n.lower().endswith(".zip")]
        if len(inner_zips) != 1:
            raise RuntimeError(
                f"Expected exactly one inner .zip in {zip_path}, found: {names}"
            )
        zf.extract(inner_zips[0], path=extract_dir)
    return os.path.join(extract_dir, inner_zips[0])


def build_tracking_record(item: dict, l1_product: str, s3_bucket: str, s3_scene_key: str) -> dict:
    """Tracking JSON for one uploaded scene. Batch items carry no `WorkflowOptions`, so
    processor_platform/version are always null here."""
    workflow_options = {
        o.get("Name"): o.get("Value") for o in item.get("WorkflowOptions", []) or []
    }
    return {
        "l0_product": item.get("InputProductReference"),
        "l1_product": l1_product,
        "workflow_name": item.get("WorkflowName", WORKFLOW_NAME),
        "processor_platform": workflow_options.get("platform"),
        "processor_version": workflow_options.get("version"),
        "source": "CDSE On-Demand Production API (BatchOrder)",
        "batch_order_id": BATCH_ORDER_ID,
        "batch_order_name": BATCH_ORDER_NAME,
        "order_item_id": item.get("Id"),
        "order_item_status": item.get("Status"),
        "order_submission_date": item.get("SubmissionDate"),
        "date_created": datetime.utcnow().isoformat() + "Z",
        "s3_bucket": s3_bucket,
        "s3_scene_key": s3_scene_key,
    }


def key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "404":
            return False
        raise  # re-raise anything that isn't "not found" (e.g. permissions)


def process_batch_item(item: dict) -> str:
    """Download, unzip, and upload a single batch item. Returns a status label; runs on a
    worker thread, so all logging goes through `log()` and all state is local to the call."""
    item_id = item.get("Id")
    item_status = str(item.get("Status", "")).lower()
    l0_reference = item.get("InputProductReference") or f"item_{item_id}"
    item_label = os.path.splitext(os.path.basename(str(l0_reference)))[0]

    if item_id is None:
        log(f"Skipping item with no Id field: {item}")
        return "skipped"
    if item_status and item_status != "completed":
        log(f"Skipping {item_label} (Id={item_id}): status={item_status}")
        return "skipped"

    download_link = item.get("DownloadLink")
    if not download_link:
        log(f"Skipping {item_label} (Id={item_id}): no DownloadLink present.")
        return "skipped"

    log(f"[{item_label}] Processing (Id={item_id})...")

    l1_product_name = None
    try:
        l1_product_name = peek_remote_inner_zip_name(download_link, item.get("ProcessedSize"))
    except Exception as e:
        log(f"[{item_label}] Could not peek remote zip contents ({e}); will determine name after download.")

    if l1_product_name is not None:
        if not l1_product_name.endswith(".zip"):
            l1_product_name += ".zip"
        s3_key = f"{S3_SCENE_UPLOAD_FOLDER.rstrip('/')}/{l1_product_name}"
        if key_exists(s3_client, S3_BUCKET, s3_key):
            log(f"[{item_label}] Already uploaded -> s3://{S3_BUCKET}/{s3_key} (skipping download/unzip/upload)")
            return "already_uploaded"

    order_zip_path = os.path.join(OUTPUT_DIR, f"{item_label}.zip")
    if not os.path.exists(order_zip_path):
        log(f"[{item_label}] Downloading order package -> {order_zip_path}")
        download_batch_item(download_link, order_zip_path)
    else:
        log(f"[{item_label}] Order package exists, download skipped -> {order_zip_path}")

    inner_zip_path = unzip_single_inner_zip(order_zip_path, OUTPUT_DIR)
    log(f"[{item_label}] Extracted product -> {inner_zip_path}")

    l1_product_name = os.path.basename(inner_zip_path)
    s3_key = f"{S3_SCENE_UPLOAD_FOLDER.rstrip('/')}/{l1_product_name}"
    if key_exists(s3_client, S3_BUCKET, s3_key):
        log(f"[{item_label}] Already uploaded -> s3://{S3_BUCKET}/{s3_key} (skipping upload)")
        return "already_uploaded"

    s3_client.upload_file(inner_zip_path, S3_BUCKET, s3_key)
    log(f"[{item_label}] Uploaded -> s3://{S3_BUCKET}/{s3_key}")

    tracking_record = build_tracking_record(item, l1_product_name, S3_BUCKET, s3_key)
    tracking_filename = os.path.splitext(l1_product_name)[0] + ".json"
    tracking_local_path = os.path.join(OUTPUT_DIR, tracking_filename)
    with open(tracking_local_path, "w") as f:
        json.dump(tracking_record, f, indent=2)

    tracking_s3_key = f"{S3_SCENE_TRACKING_UPLOAD_FOLDER.rstrip('/')}/{tracking_filename}"
    s3_client.upload_file(tracking_local_path, S3_BUCKET, tracking_s3_key)
    log(f"[{item_label}] Uploaded tracking record -> s3://{S3_BUCKET}/{tracking_s3_key}")
    return "uploaded"


os.makedirs(OUTPUT_DIR, exist_ok=True)

results = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_item = {executor.submit(process_batch_item, item): item for item in batch_items}
    for future in as_completed(future_to_item):
        item = future_to_item[future]
        item_label = item.get("InputProductReference", item.get("Id"))
        try:
            results[item_label] = future.result()
        except Exception as e:
            log(f"[{item_label}] FAILED: {e}")
            results[item_label] = "failed"

summary = {status: sum(1 for v in results.values() if v == status)
           for status in ("uploaded", "already_uploaded", "skipped", "failed")}
print(f"\nBatch download/unzip/upload complete. {summary}")

## 7. Process Scenes with ISCE3_RTC
Now the EW level-1 SLC data is available, it can be processed using the isce3_rtc workflow. Below shows an example docker command. The **scene-path** is the path to the file uploaded from this process:

```bash
docker run --env-file .env --platform linux/amd64 \
-v $PWD/data:/home/rtc_user/working \
sar-pipeline-isce3-rtc \
--scene S1A_EW_SLC__1SDH_20250101T153651_20250101T153757_057252_070AF9_1F2E \
--scene-path https://deant-data-public-dev.s3.ap-southeast-2.amazonaws.com/s1_ew_slc/data/S1A_EW_SLC__1SDH_20250101T153651_20250101T153757_057252_070AF9_1F2E.SAFE.zip \
--resolution 40 \
--skip-upload-to-s3 \
--make-existing-products \
--burst-id-list t130_253088_ew5

```